# Temporal Occupancy and Richness Trends (Thinned)

Year-wise presence-based trends by district and species occupancy.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='paper')
RANDOM_SEED = 42


In [ ]:
p = next((x for x in [Path('file6.csv'), Path('../file6.csv'), Path('../../file6.csv')] if x.exists()), None)
if p is None:
    raise FileNotFoundError('file6.csv not found')
cols = ['stateProvince','verbatimScientificName','decimalLatitude','decimalLongitude','eventDate']
df = pd.read_csv(p, usecols=cols, low_memory=False).dropna(subset=cols)
df['eventDate'] = pd.to_datetime(df['eventDate'], errors='coerce')
df['year'] = df['eventDate'].dt.year
for c in ['decimalLatitude','decimalLongitude']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.dropna(subset=['decimalLatitude','decimalLongitude','year'])


In [ ]:
lon0 = float(df['decimalLongitude'].median())
lat0 = float(df['decimalLatitude'].median())
lat_rad = np.radians(df['decimalLatitude'].to_numpy())
df['x_km'] = (df['decimalLongitude'].to_numpy() - lon0) * 111.320 * np.cos(lat_rad)
df['y_km'] = (df['decimalLatitude'].to_numpy() - lat0) * 110.574
df['grid_x'] = np.floor(df['x_km'] / 5.0).astype(int)
df['grid_y'] = np.floor(df['y_km'] / 5.0).astype(int)
df['cell_id'] = df['stateProvince'].astype(str) + '|' + df['grid_x'].astype(str) + '_' + df['grid_y'].astype(str)
df['_key'] = df['cell_id'] + '|' + df['verbatimScientificName'] + '|' + df['year'].astype(int).astype(str)
rng = np.random.default_rng(RANDOM_SEED)
df['_r'] = rng.random(len(df))
df_thin = df.sort_values('_r').groupby('_key', as_index=False).head(1)


In [ ]:
year_district = df_thin.groupby(['year','stateProvince'], as_index=False).agg(n_cells=('cell_id','nunique'),richness=('verbatimScientificName','nunique'))
year_total = df_thin.groupby('year', as_index=False).agg(n_cells=('cell_id','nunique'),richness=('verbatimScientificName','nunique'))
display(year_total.tail(15))
plt.figure(figsize=(8.2,5.2))
sns.lineplot(data=year_total, x='year', y='richness', marker='o', color='#1f618d')
plt.title('Overall annual richness (thinned by year)')
plt.tight_layout()
plt.show()


In [ ]:
top_d = year_district.groupby('stateProvince')['n_cells'].sum().nlargest(8).index
plt.figure(figsize=(10,5.5))
sns.lineplot(data=year_district[year_district['stateProvince'].isin(top_d)], x='year', y='richness', hue='stateProvince')
plt.title('Annual district richness (top sampled districts)')
plt.tight_layout()
plt.show()


In [ ]:
species_year_occ = df_thin.groupby(['year','verbatimScientificName'])['cell_id'].nunique().rename('occupied_cells').reset_index()
top_sp = species_year_occ.groupby('verbatimScientificName')['occupied_cells'].sum().nlargest(6).index
plt.figure(figsize=(10,5.5))
sns.lineplot(data=species_year_occ[species_year_occ['verbatimScientificName'].isin(top_sp)], x='year', y='occupied_cells', hue='verbatimScientificName')
plt.title('Annual occupied cells for top species')
plt.tight_layout()
plt.show()
